# Task 2.2 — Reproduce One Contribution

**Paper:** "Object Detection with Discriminatively Trained Part-Based Models"  
**Authors:** Felzenszwalb, Girshick, McAllester, Ramanan (2010)  
**Venue:** IEEE TPAMI

---

## Objective

We reproduce the core contribution of the DPM paper: comparing a **rigid (root-only) detector** against a **part-based model with deformation penalties**.

The implementation pipeline:
1. Load the toy dataset (generated in Task 2.1)
2. Define feature groups (root, parts, deformation)
3. Train a baseline rigid SVM (root features only)
4. Implement a simplified part-based scoring mechanism
5. Train a full part-based model (root + parts + deformation)
6. Evaluate and compare both models

## Step 1: Load Dataset and Define Feature Groups

This step loads the data created in Task 2.1 and defines feature slices corresponding to the DPM's root filter, part filters, and deformation costs (Section 3 of the paper).

In [ ]:
import numpy as np
import pandas as pd
from sklearn.svm import LinearSVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_classification
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

# ============================================================
# Reproducibility
# ============================================================
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# ============================================================
# Regenerate dataset (same as Task 2.1 for self-contained notebook)
# ============================================================
X_base, y = make_classification(
    n_samples=500, n_features=8, n_informative=8,
    n_redundant=0, n_clusters_per_class=2,
    class_sep=1.0, random_state=RANDOM_SEED
)

def generate_deformation_features(y, n_parts=3, deform_std_pos=0.3, deform_std_neg=1.5, seed=42):
    rng = np.random.RandomState(seed)
    deformations = np.zeros((len(y), n_parts * 2))
    for i in range(len(y)):
        if y[i] == 1:
            deformations[i] = rng.normal(0, deform_std_pos, n_parts * 2)
        else:
            deformations[i] = rng.normal(0, deform_std_neg, n_parts * 2)
    return deformations

deformation_features = generate_deformation_features(y)
X_full = np.hstack([X_base, deformation_features])

feature_names = (
    ['root_f0', 'root_f1'] +
    ['part1_f0', 'part1_f1'] +
    ['part2_f0', 'part2_f1'] +
    ['part3_f0', 'part3_f1'] +
    ['part1_dx', 'part1_dy'] +
    ['part2_dx', 'part2_dy'] +
    ['part3_dx', 'part3_dy']
)

# Define feature group indices
ROOT_IDX = [0, 1]                          # Root filter features
PART1_IDX = [2, 3]                          # Part 1 appearance
PART2_IDX = [4, 5]                          # Part 2 appearance
PART3_IDX = [6, 7]                          # Part 3 appearance
DEFORM1_IDX = [8, 9]                        # Part 1 deformation (dx, dy)
DEFORM2_IDX = [10, 11]                      # Part 2 deformation (dx, dy)
DEFORM3_IDX = [12, 13]                      # Part 3 deformation (dx, dy)
ALL_PART_IDX = PART1_IDX + PART2_IDX + PART3_IDX
ALL_DEFORM_IDX = DEFORM1_IDX + DEFORM2_IDX + DEFORM3_IDX
ALL_IDX = ROOT_IDX + ALL_PART_IDX + ALL_DEFORM_IDX

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X_full, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)

print(f"Dataset: {X_full.shape[0]} samples, {X_full.shape[1]} features")
print(f"Train: {X_train.shape[0]}, Test: {X_test.shape[0]}")
print(f"Feature groups:")
print(f"  Root:        indices {ROOT_IDX}")
print(f"  Parts:       indices {ALL_PART_IDX}")
print(f"  Deformation: indices {ALL_DEFORM_IDX}")

**What this code does:**  
Loads and structures the dataset into feature groups matching the DPM architecture. The root features correspond to the root filter $F_0$ (Section 3), part features correspond to part filters $F_1, \ldots, F_n$, and deformation features correspond to the displacement penalty $d_i \cdot \Phi_d(dx, dy)$ from Equation 3.

---

## Step 2: Feature Extraction (Analogy to HOG)

In the original paper, HOG features are extracted from images at multiple scales. Here, we work with pre-computed synthetic features. We standardize them as a preprocessing step (analogous to HOG normalization described in Section 2.4).

In [ ]:
# Standardize features (analogous to HOG normalization in Section 2.4)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Feature statistics after scaling (train set):")
print(f"  Mean: {X_train_scaled.mean(axis=0).round(3)}")
print(f"  Std:  {X_train_scaled.std(axis=0).round(3)}")

**What this code does:**  
Standardizes features to zero mean and unit variance. In the paper, HOG features undergo block normalization (Section 2.4) to achieve illumination invariance. Our standardization serves an analogous purpose.

---

## Step 3: Train Baseline Rigid SVM Model (Root-Only)

This corresponds to the Dalal & Triggs baseline — a linear SVM trained on only the root (global) features, without any part information.

**Paper correspondence:** Section 6, Table 3 — the root-only model row.

In [ ]:
# ============================================================
# Model 1: Rigid Baseline (Root-Only SVM)
# Corresponds to Dalal & Triggs HOG detector
# Uses only root features (global appearance template)
# ============================================================

# Extract root features only
X_train_root = X_train_scaled[:, ROOT_IDX]
X_test_root = X_test_scaled[:, ROOT_IDX]

# Train linear SVM (analogous to the paper's linear filter scoring)
svm_root = LinearSVC(C=1.0, max_iter=10000, random_state=RANDOM_SEED)
svm_root.fit(X_train_root, y_train)

# Predict and evaluate
y_pred_root = svm_root.predict(X_test_root)
acc_root = accuracy_score(y_test, y_pred_root)

print("=" * 50)
print("MODEL 1: Rigid Baseline (Root-Only SVM)")
print("=" * 50)
print(f"Features used: {[feature_names[i] for i in ROOT_IDX]}")
print(f"Number of features: {len(ROOT_IDX)}")
print(f"Accuracy: {acc_root:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred_root, target_names=['Background', 'Object']))

**What this code does:**  
Trains a linear SVM using only root features (2 features), simulating a rigid template detector. This is the baseline model — equivalent to applying the root filter $F_0$ alone without any part filters. The paper shows (Section 6, Table 3) that the root-only model achieves significantly lower AP than the full model, and we expect similar behavior here.

---

## Step 4: Implement Simplified Part-Based Scoring Mechanism

The DPM's scoring function (Equation 2 in the paper) is:

$$\text{score} = \underbrace{F_0 \cdot \phi_{\text{root}}}_{\text{root score}} + \sum_{i=1}^{n} \left[ \underbrace{F_i \cdot \phi_{\text{part}_i}}_{\text{part score}} - \underbrace{d_i \cdot \Phi_d(dx_i, dy_i)}_{\text{deformation cost}} \right] + b$$

We simulate this by constructing combined features that include:
- Root features (root filter response)
- Part appearance features (part filter responses)
- Squared deformation features (quadratic deformation penalty)

The quadratic deformation cost from Equation 3 uses $\Phi_d(dx, dy) = (dx, dy, dx^2, dy^2)$. We add the squared displacement terms as features.

In [ ]:
# ============================================================
# DPM-inspired feature construction
# Add quadratic deformation terms: dx^2, dy^2 for each part
# This simulates Equation 3: d_i * (dx, dy, dx^2, dy^2)
# ============================================================

def construct_dpm_features(X, root_idx, part_idx, deform_idx):
    """
    Construct DPM-style features:
    - Root features (root filter response)
    - Part features (part filter responses)  
    - Deformation features: (dx, dy, dx^2, dy^2) for each part
    
    Corresponds to the feature vector beta^T * Phi(x) from Section 3.
    """
    root = X[:, root_idx]
    parts = X[:, part_idx]
    deform_raw = X[:, deform_idx]
    
    # Add squared deformation terms (Equation 3: quadratic penalty)
    deform_squared = deform_raw ** 2
    
    # Full DPM feature vector: [root, parts, dx, dy, dx^2, dy^2]
    return np.hstack([root, parts, deform_raw, deform_squared])

# Construct DPM features
X_train_dpm = construct_dpm_features(X_train_scaled, ROOT_IDX, ALL_PART_IDX, ALL_DEFORM_IDX)
X_test_dpm = construct_dpm_features(X_test_scaled, ROOT_IDX, ALL_PART_IDX, ALL_DEFORM_IDX)

dpm_feature_names = (
    [feature_names[i] for i in ROOT_IDX] +
    [feature_names[i] for i in ALL_PART_IDX] +
    [feature_names[i] for i in ALL_DEFORM_IDX] +
    ['part1_dx2', 'part1_dy2', 'part2_dx2', 'part2_dy2', 'part3_dx2', 'part3_dy2']
)

print(f"DPM feature vector size: {X_train_dpm.shape[1]}")
print(f"Breakdown:")
print(f"  Root features:       {len(ROOT_IDX)}")
print(f"  Part features:       {len(ALL_PART_IDX)}")
print(f"  Deformation (dx,dy): {len(ALL_DEFORM_IDX)}")
print(f"  Deformation (dx²,dy²): {len(ALL_DEFORM_IDX)}")
print(f"  Total:               {X_train_dpm.shape[1]}")

**What this code does:**  
Constructs the DPM-style feature vector by combining root, part, and deformation features — including quadratic (squared) displacement terms. This directly implements the feature representation from Section 3, Equation 3 of the paper, where the deformation penalty is $d_i \cdot (dx, dy, dx^2, dy^2)$. The linear SVM will learn the deformation coefficients $d_i$ as part of its weight vector.

---

## Step 5: Train Part-Based Model with Deformation Features

In [ ]:
# ============================================================
# Model 2: Part-Based Model (DPM-style)
# Uses root + part + deformation features
# Corresponds to full model from Section 3
# ============================================================

svm_dpm = LinearSVC(C=1.0, max_iter=10000, random_state=RANDOM_SEED)
svm_dpm.fit(X_train_dpm, y_train)

y_pred_dpm = svm_dpm.predict(X_test_dpm)
acc_dpm = accuracy_score(y_test, y_pred_dpm)

print("=" * 50)
print("MODEL 2: Part-Based Model (DPM-style)")
print("=" * 50)
print(f"Number of features: {X_train_dpm.shape[1]}")
print(f"Accuracy: {acc_dpm:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred_dpm, target_names=['Background', 'Object']))

**What this code does:**  
Trains the full part-based model using all DPM-style features. The linear SVM learns weights for root features (root filter $F_0$), part features (part filters $F_i$), and deformation features (deformation coefficients $d_i$). This corresponds to the jointly trained model from Section 4.

Note: The original paper uses Latent SVM (Section 4) where part positions are latent variables optimized during training. Our simplified version treats part features as observed, which is equivalent to having already solved the latent variable placement (the "relabeling" step in the paper's coordinate descent algorithm).

---

## Step 6: Implement Simplified Hard Negative Mining

The paper (Section 4.2) uses hard negative mining to focus training on difficult background examples. We simulate this by identifying misclassified negatives and retraining with emphasis on them.

In [ ]:
# ============================================================
# Simplified Hard Negative Mining (Section 4.2)
# 1. Train initial model
# 2. Find hard negatives (false positives on training set)
# 3. Retrain with hard negatives upweighted
# ============================================================

def hard_negative_mining(X_train, y_train, X_test, y_test, n_rounds=3, C=1.0):
    """Simplified hard negative mining following Section 4.2."""
    
    sample_weights = np.ones(len(y_train))
    results = []
    
    for round_num in range(n_rounds):
        # Train SVM with current sample weights
        svm = LinearSVC(C=C, max_iter=10000, random_state=RANDOM_SEED)
        svm.fit(X_train, y_train, sample_weight=sample_weights)
        
        # Find hard negatives: negative samples misclassified as positive
        y_train_pred = svm.predict(X_train)
        hard_neg_mask = (y_train == 0) & (y_train_pred == 1)
        n_hard_neg = hard_neg_mask.sum()
        
        # Upweight hard negatives for next round
        sample_weights[hard_neg_mask] *= 2.0
        
        # Evaluate on test set
        y_test_pred = svm.predict(X_test)
        acc = accuracy_score(y_test, y_test_pred)
        
        results.append({
            'round': round_num + 1,
            'hard_negatives': n_hard_neg,
            'test_accuracy': acc
        })
        print(f"Round {round_num + 1}: Hard negatives = {n_hard_neg}, Test accuracy = {acc:.4f}")
    
    return svm, results

print("Hard Negative Mining (DPM-style, Section 4.2)")
print("=" * 50)
svm_hnm, hnm_results = hard_negative_mining(X_train_dpm, y_train, X_test_dpm, y_test)
y_pred_hnm = svm_hnm.predict(X_test_dpm)
acc_hnm = accuracy_score(y_test, y_pred_hnm)

**What this code does:**  
Implements a simplified version of the hard negative mining from Section 4.2. The paper iteratively finds false positives (background windows that score above threshold), adds them to the training set, and retrains. Our simplified version upweights misclassified negatives instead of maintaining an explicit cache, achieving a similar effect of focusing the model on difficult examples.

---

## Step 7: Compare All Models

In [ ]:
# ============================================================
# Comparison summary
# ============================================================

print("\n" + "=" * 60)
print("MODEL COMPARISON SUMMARY")
print("=" * 60)
print(f"{'Model':<35} {'Accuracy':>10} {'# Features':>12}")
print("-" * 60)
print(f"{'1. Rigid Baseline (Root-Only)':<35} {acc_root:>10.4f} {len(ROOT_IDX):>12}")
print(f"{'2. Part-Based Model (DPM)':<35} {acc_dpm:>10.4f} {X_train_dpm.shape[1]:>12}")
print(f"{'3. DPM + Hard Negative Mining':<35} {acc_hnm:>10.4f} {X_train_dpm.shape[1]:>12}")
print("-" * 60)
print(f"\nImprovement (DPM over Rigid): {(acc_dpm - acc_root)*100:+.1f} percentage points")
print(f"Improvement (DPM+HNM over Rigid): {(acc_hnm - acc_root)*100:+.1f} percentage points")

In [ ]:
# ============================================================
# Visualize learned model weights
# Shows which features the DPM model relies on most
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Root model weights
root_weights = svm_root.coef_[0]
axes[0].barh([feature_names[i] for i in ROOT_IDX], root_weights, color='steelblue')
axes[0].set_title('Rigid Baseline: Learned Weights (Root Filter Only)')
axes[0].set_xlabel('Weight')
axes[0].axvline(x=0, color='black', linewidth=0.5)

# DPM model weights  
dpm_weights = svm_dpm.coef_[0]
colors = (['steelblue'] * 2 + ['coral'] * 6 + ['seagreen'] * 6 + ['mediumpurple'] * 6)
axes[1].barh(dpm_feature_names, dpm_weights, color=colors)
axes[1].set_title('Part-Based Model: Learned Weights')
axes[1].set_xlabel('Weight')
axes[1].axvline(x=0, color='black', linewidth=0.5)

# Add legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='steelblue', label='Root (F₀)'),
    Patch(facecolor='coral', label='Parts (Fᵢ)'),
    Patch(facecolor='seagreen', label='Deformation (dᵢ · Φ_d)'),
    Patch(facecolor='mediumpurple', label='Deformation² (dᵢ · Φ_d²)')
]
axes[1].legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
os.makedirs('results', exist_ok=True)
plt.savefig('results/model_weights_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: results/model_weights_comparison.png")

**What this code does:**  
Visualizes the learned SVM weight vectors for both models. In the DPM model, the weights on root features correspond to root filter $F_0$, weights on part features correspond to part filters $F_i$, and weights on deformation features correspond to deformation coefficients $d_i$ from Equation 3. Negative weights on squared deformation terms indicate the model learned to **penalize** large displacements, matching the paper's constraint that deformation coefficients should be non-positive (Section 3, below Equation 3).

---

## Step 8: Analyze the Scoring Function

In [ ]:
# ============================================================  
# Decompose the DPM score into components
# score = root_score + part_scores - deformation_costs + bias
# This mirrors Equation 2 in the paper
# ============================================================

weights = svm_dpm.coef_[0]
bias = svm_dpm.intercept_[0]

# Decompose score for each test sample
root_scores = X_test_dpm[:, :2] @ weights[:2]
part_scores = X_test_dpm[:, 2:8] @ weights[2:8]
deform_linear_scores = X_test_dpm[:, 8:14] @ weights[8:14]
deform_quad_scores = X_test_dpm[:, 14:20] @ weights[14:20]
total_scores = svm_dpm.decision_function(X_test_dpm)

# Visualize score decomposition
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, scores, title in zip(
    axes.flat,
    [root_scores, part_scores, deform_linear_scores + deform_quad_scores, total_scores],
    ['Root Score (F₀·φ_root)', 'Part Scores (Σ Fᵢ·φ_partᵢ)', 
     'Deformation Cost (−Σ dᵢ·Φ_d)', 'Total Score']
):
    ax.hist(scores[y_test == 0], bins=20, alpha=0.6, label='Background', color='blue')
    ax.hist(scores[y_test == 1], bins=20, alpha=0.6, label='Object', color='red')
    ax.set_title(title)
    ax.set_xlabel('Score')
    ax.set_ylabel('Count')
    ax.legend()
    ax.axvline(x=0, color='black', linestyle='--', linewidth=0.5)

plt.suptitle('DPM Score Decomposition (Paper Equation 2)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('results/score_decomposition.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: results/score_decomposition.png")

**What this code does:**  
Decomposes the DPM decision function into its constituent components — root score, part scores, and deformation cost — and visualizes the distribution for positive vs. negative examples. This directly corresponds to Equation 2 in the paper:

$$\text{score} = F_0 \cdot \phi_{\text{root}} + \sum_{i=1}^{n}[F_i \cdot \phi_{\text{part}_i} - d_i \cdot \Phi_d(dx_i, dy_i)] + b$$

The visualization shows that each component contributes to class separation, and the combined score achieves the best discrimination — validating the DPM's design that root + parts + deformation are all beneficial.